## Using AI in python

Using AI in python is quite straightforward, many frameworks exists that simplifies calling AI models throught API.

Some notable providers of services are:
- OpenAI : Creators of the GPT-X and oX series
- Google : Creators of the Gemini series
- Mistral : Creators of the Mistral series
- XAI : Creators of the Groq series

Some labs also provide Open source models, these models can be found on HuggingFace
- Meta : Creators of the Llama series
- Alibaba : Creators of the Qwen series (one of the best opensource models)
- Tencent : Creators of the Hunyuan series

Worth noting, but when considering what model to use, you should consider the following:
- Pricing: The cost is expressed in $ per million tokens
- Speed : The speed of inference
- Rate limits : some providers are rate limiting to avoid overload, which can be a burden
- Performance : Not all use cases need frontier models

There are also some agregators that serve some open source models, like OpenRouter

For beginner, most providers are offering a free tier, in order to be able to test the models without too much trouble.

Last but not least, the serving of LLM is very hardware dependant, some specialised actors like Cerebra or Groq are offering very fast inference (> 2000 tokens / seconds)


### What is an API

An API, or Application Programming Interface, is like a menu in a restaurant. It lets different computer programs talk to each other and share information. Imagine you want to order food from the kitchen (another program). The menu (API) tells you what's available and how to ask for it. An API token is like a special key that you need to show to the waiter (the API) before they'll take your order. It helps make sure that only authorized programs can access the API and its data.

For this notebooks, we are first going to create an api key on AI Studio

In [ ]:
import os

## This command will add the API key to your environment variables
os.environ['GEMINI_API_KEY']=""

gemini_model = "gemini/gemini-2.0-flash" # syntax for litellm is "provider/model"

## First llm inference

Go on https://mirascope.com/docs/mirascope and check the documentation to install the library with the litellm dependancy.

Start by installing the mirascope library with pip

In [2]:
!pip install "mirascope[litellm]"

In [4]:
from mirascope import llm
from mirascope.core import prompt_template

@llm.call(provider="litellm", model=gemini_model, stream=True)
@prompt_template("""SYSTEM:
You are a helpful assistant that greats people on demand.

USER:
Name of the person to greet: {name}
""")
def say_hey(name: str) -> str: ...

In [5]:
answer = ""
name = "Gabriel"

for chunk, _ in say_hey(name=name):
    answer += chunk.content
    print(chunk.content, end="", flush=True)

Hello Gabriel, it's a pleasure to meet you! How can I help you today?


### **Problem 1**
Create a function to translate some text from any language to any language.

The function should have argument:
- text_to_translate
- target_language

And output the translated text.

Try it on a wikipedia page.

In [6]:
@llm.call(provider="litellm", model=gemini_model, stream=True)
@prompt_template("""SYSTEM:
## Role
You are an expert translator that translates text to a target language.

## Instructions
Based on an input text provided as TEXT_TO_TRANSLATE, translate it to the language specified as TARGET_LANGUAGE.

## Constraints
- Do not change the meaning of the text.
- Keep the same text structure.
- Only translate the text, do not add any additional information.

USER:
## TEXT_TO_TRANSLATE
{text}

## TARGET_LANGUAGE
{target_language}
""")
def translate(text: str, target_language: str) -> str: ...

In [7]:
answer = ""

text = """Hautespen naturala indibiduoen biziraupen diferentziala eta ugalketa, fenotipoan dauden desberdintasunak direla eta. Eboluzioaren mekanismo giltzarria da, belaunaldiz belaunaldi populazio baten karaktere hereditarioak aldatzea. Charles Darwinek “hautespen natural” terminoa zabaldu zuen, eta hautespen artifizialarekin alderatu zuen, zeina, bere ustez, intentziozkoa baita; hautespen naturala, aldiz, ez."""
target_language = "French"

for chunk, _ in translate(text=text, target_language=target_language):
    answer += chunk.content
    print(chunk.content, end="", flush=True)

La sélection naturelle est la survie et la reproduction différentielles des individus en raison de différences de phénotype. C'est un mécanisme clé de l'évolution, l'altération des caractères héréditaires d'une population de génération en génération. Le terme « sélection naturelle » a été popularisé par Charles Darwin, qui l'a comparé à la sélection artificielle, qui, selon lui, est intentionnelle, contrairement à la sélection naturelle.


### **Problem 2**
Create a function to summarize some text

The function should have argument:
- text
- key_attention_point

And output the text summary.
Try it on a wikipedia page as well

In [9]:
@llm.call(provider="litellm", model=gemini_model, stream=True)
@prompt_template("""SYSTEM:
## Role
You are an expert summarizer that summarizes text based on a key point of interest.

## Instructions
Based on an input text provided as TEXT_TO_SUMMARIZE, summarize it focusing on the key point of interest specified as INTEREST.

## Constraints
- Output the summary as a poem
- Always answer in English

USER:
## TEXT_TO_SUMMARIZE
{text}

## INTEREST
{key_point_of_interest}
""")
def summarize(text: str, key_point_of_interest: str) -> str: ...

In [ ]:
answer = ""

text = """Hautespen naturala indibiduoen biziraupen diferentziala eta ugalketa, fenotipoan dauden desberdintasunak direla eta. Eboluzioaren mekanismo giltzarria da, belaunaldiz belaunaldi populazio baten karaktere hereditarioak aldatzea. Charles Darwinek “hautespen natural” terminoa zabaldu zuen, eta hautespen artifizialarekin alderatu zuen, zeina, bere ustez, intentziozkoa baita; hautespen naturala, aldiz, ez."""
key_point_of_interest = "Focus on peoples"

for chunk, _ in summarize(text=text, key_point_of_interest=key_point_of_interest):
    answer += chunk.content
    print(chunk.content, end="", flush=True)

In nature's grand design,
A dance of traits, where some outshine.
Not by intent, but chance's hand,
Survival's path, across the land.

Though people may not guide the way,
Their presence felt, in disarray,
On species shaped, by needs and strife,
A tapestry woven, of fragile life.


### **Problem 3**
Check the mirascope documentation on how to use multimodal inputs, and create a function that captions an image, provided as a URL

In [12]:
@llm.call(provider="litellm", model=gemini_model, stream=True)
@prompt_template("""SYSTEM:
## Role
You are an expert image captionner that describes the content of an image based on its URL.

## Instructions
Based on an input image provided as IMAGE_URL, describe the content of the image in a concise manner.

## Constraints
- Use bullet point.
- Describe both the scene and the style of the image.
- Always answer in English.

USER:
## IMAGE
{image_url:image}

""")
def captionning(image_url: str) -> str: ...

In [14]:
answer = ""

image_url="https://xtensio.com/wp-content/uploads/2019/04/Thumb-Customer-Support.jpg.webp"

for chunk, _ in captionning(image_url=image_url):
    answer += chunk.content
    print(chunk.content, end="", flush=True)

Here are the bullet point descriptions of the image:
*   **Scene:** The image shows a customer persona named Jack Rowland. It includes a profile picture, age, job, family status, location, and a character description. It also provides a bio, motivations, goals, personality traits, preferred channels, and frustrations.
*   **Style:** The image is designed in a modern, infographic style, with a clean layout, clear typography, and a consistent color scheme of white and teal. It uses bar graphs to visualize the levels of motivation and preferred channels. Personality traits are represented on a sliding scale from introvert to extrovert. The overall style is professional and easy to understand.

### **Problem 4**
Check the documentation for:
- `input` method, which is a native python method
- `MESSAGES` role in mirascope, and build a simple turn by turn chatbot in jupyter

### **BONUS**
Download LM Studio in python, download the model Qwen3 0.6B and find a way to use it with mirascope by changing the openai api base and api key

Hints: For that you will need to:
- Download LMStudio
- Download lmstudio-community/Qwen3-0.6B-GGUF model
- Enable Lm studio api
- Check the documention of litellm on how to call a LM studio model
- Build the funciton to call it

In [ ]:
os.environ['OPENAI_API_BASE']=""
os.environ['OPENAI_API_KEY']="sk_1234"